In [0]:
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd
import xml.etree.ElementTree as ET
import urllib3
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import StringType

# ============================================================
# Schema — SAP OData → Databricks Bronze
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    NullType, StringType, DateType, 
    DecimalType, IntegerType, LongType, TimestampType
)
from pyspark.sql.functions import col, to_date, current_timestamp, lit

 
vbap_schema = {
    "Vbeln"                          : StringType(),
    "Posnr"                          : StringType(),        # NUMC
    "Matnr"                          : StringType(),
    "Matwa"                          : StringType(),
    "Pmatn"                          : StringType(),
    "Charg"                          : StringType(),
    "Matkl"                          : StringType(),
    "Arktx"                          : StringType(),
    "Pstyv"                          : StringType(),
    "Posar"                          : StringType(),
    "Lfrel"                          : StringType(),
    "Fkrel"                          : StringType(),
    "Uepos"                          : StringType(),        # NUMC
    "Grpos"                          : StringType(),        # NUMC
    "Abgru"                          : StringType(),
    "Prodh"                          : StringType(),
    "Zwert"                          : DecimalType(23, 2),  # CURR
    "Zmeng"                          : DecimalType(15, 3),  # QUAN
    "Zieme"                          : StringType(),        # UNIT
    "Umziz"                          : DecimalType(15, 3),  # DEC
    "Umzin"                          : DecimalType(15, 3),  # DEC
    "Meins"                          : StringType(),        # UNIT
    "Smeng"                          : DecimalType(15, 3),  # QUAN
    "Ablfz"                          : DecimalType(15, 3),  # QUAN
    "Abdat"                          : DateType(),          # DATS
    "Absfz"                          : DecimalType(15, 3),  # QUAN
    "Posex"                          : StringType(),
    "Kdmat"                          : StringType(),
    "Kbver"                          : DecimalType(15, 3),  # DEC
    "Kever"                          : DecimalType(15, 3),  # DEC
    "Vkgru"                          : StringType(),
    "Vkaus"                          : StringType(),
    "Grkor"                          : StringType(),        # NUMC
    "Fmeng"                          : StringType(),
    "Uebtk"                          : StringType(),
    "Uebto"                          : DecimalType(15, 3),  # DEC
    "Untto"                          : DecimalType(15, 3),  # DEC
    "Faksp"                          : StringType(),
    "Atpkz"                          : StringType(),
    "Rkfkf"                          : StringType(),
    "Spart"                          : StringType(),
    "Gsber"                          : StringType(),
    "Netwr"                          : DecimalType(23, 2),  # CURR
    "Waerk"                          : StringType(),        # CUKY
    "Antlf"                          : DecimalType(15, 3),  # DEC
    "Kztlf"                          : StringType(),
    "Chspl"                          : StringType(),
    "Kwmeng"                         : DecimalType(15, 3),  # QUAN
    "Lsmeng"                         : DecimalType(15, 3),  # QUAN
    "Kbmeng"                         : DecimalType(15, 3),  # QUAN
    "Klmeng"                         : DecimalType(15, 3),  # QUAN
    "Vrkme"                          : StringType(),        # UNIT
    "Umvkz"                          : DecimalType(15, 3),  # DEC
    "Umvkn"                          : DecimalType(15, 3),  # DEC
    "Brgew"                          : DecimalType(15, 3),  # QUAN
    "Ntgew"                          : DecimalType(15, 3),  # QUAN
    "Gewei"                          : StringType(),        # UNIT
    "Volum"                          : DecimalType(15, 3),  # QUAN
    "Voleh"                          : StringType(),        # UNIT
    "Vbelv"                          : StringType(),
    "Posnv"                          : StringType(),        # NUMC
    "Vgbel"                          : StringType(),
    "Vgpos"                          : StringType(),        # NUMC
    "Voref"                          : StringType(),
    "Upflu"                          : StringType(),
    "Erlre"                          : StringType(),
    "Lprio"                          : StringType(),        # NUMC
    "Werks"                          : StringType(),
    "Lgort"                          : StringType(),
    "Vstel"                          : StringType(),
    "Route"                          : StringType(),
    "Stkey"                          : StringType(),
    "Stdat"                          : DateType(),          # DATS
    "Stlnr"                          : StringType(),
    "Stpos"                          : DecimalType(15, 3),  # DEC
    "Awahr"                          : StringType(),        # NUMC
    "Erdat"                          : DateType(),          # DATS — Created On
    "Ernam"                          : StringType(),
    "Erzet"                          : StringType(),        # TIMS — Created Time
    "Taxm1"                          : StringType(),
    "Taxm2"                          : StringType(),
    "Taxm3"                          : StringType(),
    "Taxm4"                          : StringType(),
    "Taxm5"                          : StringType(),
    "Taxm6"                          : StringType(),
    "Taxm7"                          : StringType(),
    "Taxm8"                          : StringType(),
    "Taxm9"                          : StringType(),
    "Vbeaf"                          : DecimalType(15, 3),  # DEC
    "Vbeav"                          : DecimalType(15, 3),  # DEC
    "Vgref"                          : StringType(),
    "Netpr"                          : DecimalType(23, 2),  # CURR
    "Kpein"                          : DecimalType(15, 3),  # DEC
    "Kmein"                          : StringType(),        # UNIT
    "Shkzg"                          : StringType(),
    "Sktof"                          : StringType(),
    "Mtvfp"                          : StringType(),
    "Sumbd"                          : StringType(),
    "Kondm"                          : StringType(),
    "Ktgrm"                          : StringType(),
    "Bonus"                          : StringType(),
    "Provg"                          : StringType(),
    "Eannr"                          : StringType(),
    "Prsok"                          : StringType(),
    "Bwtar"                          : StringType(),
    "Bwtex"                          : StringType(),
    "Xchpf"                          : StringType(),
    "Xchar"                          : StringType(),
    "Lfmng"                          : DecimalType(15, 3),  # QUAN
    "Stafo"                          : StringType(),
    "Wavwr"                          : DecimalType(23, 2),  # CURR
    "Kzwi1"                          : DecimalType(23, 2),  # CURR
    "Kzwi2"                          : DecimalType(23, 2),  # CURR
    "Kzwi3"                          : DecimalType(23, 2),  # CURR
    "Kzwi4"                          : DecimalType(23, 2),  # CURR
    "Kzwi5"                          : DecimalType(23, 2),  # CURR
    "Kzwi6"                          : DecimalType(23, 2),  # CURR
    "Stcur"                          : DecimalType(15, 3),  # DEC
    "Aedat"                          : DateType(),          # DATS — Last Changed Date
    "Ean11"                          : StringType(),
    "Fixmg"                          : StringType(),
    "Prctr"                          : StringType(),
    "Mvgr1"                          : StringType(),
    "Mvgr2"                          : StringType(),
    "Mvgr3"                          : StringType(),
    "Mvgr4"                          : StringType(),
    "Mvgr5"                          : StringType(),
    "Kmpmg"                          : DecimalType(15, 3),  # QUAN
    "Sugrd"                          : StringType(),
    "Sobkz"                          : StringType(),
    "Vpzuo"                          : StringType(),
    "Paobjnr"                        : StringType(),
    "PsPspPnr"                       : StringType(),        # NUMC
    "Aufnr"                          : StringType(),
    "Vpmat"                          : StringType(),
    "Vpwrk"                          : StringType(),
    "Prbme"                          : StringType(),        # UNIT
    "Umref"                          : DecimalType(15, 6),  # FLTP
    "Knttp"                          : StringType(),
    "Kzvbr"                          : StringType(),
    "Sernr"                          : StringType(),
    "Objnr"                          : StringType(),
    "Abgrs"                          : StringType(),
    "Bedae"                          : StringType(),
    "Cmpre"                          : DecimalType(23, 2),  # CURR
    "Cmtfg"                          : StringType(),
    "Cmpnt"                          : StringType(),
    "Cmkua"                          : DecimalType(15, 3),  # DEC
    "Cuobj"                          : StringType(),        # NUMC
    "CuobjCh"                        : StringType(),        # NUMC
    "Cepok"                          : StringType(),
    "Koupd"                          : StringType(),
    "Serail"                         : StringType(),
    "Anzsn"                          : IntegerType(),       # INT4
    "Nachl"                          : StringType(),
    "Magrv"                          : StringType(),
    "Mprok"                          : StringType(),
    "Vgtyp"                          : StringType(),
    "Prosa"                          : StringType(),
    "Uepvw"                          : StringType(),
    "Kalnr"                          : StringType(),        # NUMC
    "Klvar"                          : StringType(),
    "Sposn"                          : StringType(),
    "Kowrr"                          : StringType(),
    "Stadat"                         : DateType(),          # DATS
    "Exart"                          : StringType(),
    "Prefe"                          : StringType(),
    "Knumh"                          : StringType(),
    "Clint"                          : StringType(),        # NUMC
    "Chmvs"                          : StringType(),        # NUMC
    "Stlty"                          : StringType(),
    "Stlkn"                          : StringType(),        # NUMC
    "Stpoz"                          : StringType(),        # NUMC
    "Stman"                          : StringType(),
    "ZschlK"                         : StringType(),
    "KalsmK"                         : StringType(),
    "Kalvar"                         : StringType(),
    "Kosch"                          : StringType(),
    "Upmat"                          : StringType(),
    "Ukonm"                          : StringType(),
    "Mfrgr"                          : StringType(),
    "Plavo"                          : StringType(),
    "Kannr"                          : StringType(),
    "CmpreFlt"                       : DecimalType(15, 6),  # FLTP
    "Abfor"                          : StringType(),
    "Abges"                          : DecimalType(15, 6),  # FLTP
    "J1bcfop"                        : StringType(),
    "J1btaxlw1"                      : StringType(),
    "J1btaxlw2"                      : StringType(),
    "J1btxsdc"                       : StringType(),
    "Wktnr"                          : StringType(),
    "Wktps"                          : StringType(),        # NUMC
    "Skopf"                          : StringType(),
    "Kzbws"                          : StringType(),
    "Wgru1"                          : StringType(),
    "Wgru2"                          : StringType(),
    "KnumaPi"                        : StringType(),
    "KnumaAg"                        : StringType(),
    "Kzfme"                          : StringType(),
    "Lstanr"                         : StringType(),
    "Techs"                          : StringType(),
    "Mwsbp"                          : DecimalType(23, 2),  # CURR
    "Berid"                          : StringType(),
    "Pctrf"                          : StringType(),
    "LogsysExt"                      : StringType(),
    "J1btaxlw3"                      : StringType(),
    "J1btaxlw4"                      : StringType(),
    "J1btaxlw5"                      : StringType(),
    "Stockloc"                       : StringType(),
    "Sloctype"                       : StringType(),
    "MsrRetReason"                   : StringType(),
    "MsrRefundCode"                  : StringType(),
    "MsrApprovBlock"                 : StringType(),
    "NrabKnumh"                      : StringType(),
    "TrmriskRelevant"                : StringType(),
    "SgtRcat"                        : StringType(),
    "VbkdPosnr"                      : StringType(),        # NUMC
    "VedaPosnr"                      : StringType(),        # NUMC
    "Handoverloc"                    : StringType(),
    "ExtRefItemId"                   : StringType(),
    "Handoverdate"                   : DateType(),          # DATS
    "Handovertime"                   : StringType(),        # TIMS
    "TcAutDet"                       : StringType(),
    "ManualTcReason"                 : StringType(),
    "FiscalIncentive"                : StringType(),
    "TaxSubjectSt"                   : StringType(),
    "FiscalIncentiveId"              : StringType(),
    "Spcsto"                         : StringType(),        # NUMC
    "RevaccRefid"                    : StringType(),
    "RevaccReftype"                  : StringType(),
    "Dataaging"                      : DateType(),          # DATS
    "Absta"                          : StringType(),
    "Besta"                          : StringType(),
    "Cmppi"                          : StringType(),
    "Cmppj"                          : StringType(),
    "Costa"                          : StringType(),
    "Dcsta"                          : StringType(),
    "Fksaa"                          : StringType(),
    "Fssta"                          : StringType(),
    "Gbsta"                          : StringType(),
    "Lfgsa"                          : StringType(),
    "Lfsta"                          : StringType(),
    "Lssta"                          : StringType(),
    "Manek"                          : StringType(),
    "Rfgsa"                          : StringType(),
    "Rfsta"                          : StringType(),
    "Uvall"                          : StringType(),
    "Uvfak"                          : StringType(),
    "Uvprs"                          : StringType(),
    "Uvvlk"                          : StringType(),
    "Uvp01"                          : StringType(),
    "Uvp02"                          : StringType(),
    "Uvp03"                          : StringType(),
    "Uvp04"                          : StringType(),
    "Uvp05"                          : StringType(),
    "Wbsta"                          : StringType(),
    "Emcst"                          : StringType(),
    "Slcst"                          : StringType(),
    "TotalLccst"                     : StringType(),
    "Pcsta"                          : StringType(),
    "ReqqtyBu"                       : DecimalType(15, 3),  # QUAN
    "Handle"                         : StringType(),
    "PbsState"                       : StringType(),
    "Ifrs15Relevance"                : StringType(),
    "Ifrs15TotalSsp"                 : DecimalType(23, 2),  # CURR
    "Revfp"                          : StringType(),
    "CappedNetAmount"                : DecimalType(23, 2),  # CURR
    "CappedNetAmountAlertThld"       : StringType(),        # NUMC
    "SessionCreationDate"            : DateType(),          # DATS
    "SessionCreationTime"            : StringType(),        # TIMS
    "OriginalPlant"                  : StringType(),
    "AtpAbcSubstitutionStatus"       : StringType(),
    "DummySlsdocitemInclEewPs"       : StringType(),
    "PoQuan"                         : DecimalType(15, 3),  # QUAN
    "PoUnit"                         : StringType(),        # UNIT
    "MillSeGposn"                    : StringType(),        # NUMC
    "MillBatchSelF"                  : StringType(),
    "GloLogRef1It"                   : StringType(),
    "CpdUpdat"                       : StringType(),  
    "Zapcgki"                        : StringType(),        # NUMC
    "ApcgkExtendi"                   : StringType(),        # NUMC
    "Zabdati"                        : DateType(),          # DATS
    "AufplOlc"                       : StringType(),        # NUMC
    "AplzlOlc"                       : StringType(),        # NUMC
    "Ad01profnr"                     : StringType(),
    "Admoi"                          : StringType(),
    "Adicc"                          : StringType(),        # NUMC
    "Adpri"                          : StringType(),
    "Addns"                          : StringType(),
    "Adacn"                          : StringType(),
    "Labsg"                          : StringType(),
    "Fabsg"                          : StringType(),
    "PrLL"                           : StringType(),
    "PrFF"                           : StringType(),
    "PrFL"                           : StringType(),
    "FercInd"                        : StringType(),
    "FshSeasonYear"                  : StringType(),
    "FshSeason"                      : StringType(),
    "FshCollection"                  : StringType(),
    "FshTheme"                       : StringType(),
    "FshCrsd"                        : StringType(),
    "FshSearef"                      : StringType(),
    "FshCandate"                     : DateType(),          # DATS
    "FshPsmPfmSplit"                 : StringType(),
    "FshVasRel"                      : StringType(),
    "FshVasPrntId"                   : StringType(),        # NUMC
    "FshTransaction"                 : StringType(),
    "FshItemGroup"                   : StringType(),        # NUMC
    "FshItem"                        : StringType(),        # NUMC
    "FshVasref"                      : StringType(),
    "FshGridCondRec"                 : StringType(),
    "FshPqrUepos"                    : StringType(),        # NUMC
    "Kostl"                          : StringType(),
    "Fonds"                          : StringType(),
    "Fistl"                          : StringType(),
    "Fkber"                          : StringType(),
    "GrantNbr"                       : StringType(),
    "BudgetPd"                       : StringType(),
    "IuidRelevant"                   : StringType(),
    "Equnr"                          : StringType(),
    "Eqart"                          : StringType(),
    "J3glvart"                       : StringType(),
    "J3gdatvo"                       : DateType(),          # DATS
    "J3gdatbi"                       : DateType(),          # DATS
    "J3gbelnri"                      : StringType(),
    "J3gposnri"                      : StringType(),        # NUMC
    "PrsObjnr"                       : StringType(),
    "PrsSdSpsnr"                     : StringType(),        # NUMC
    "PrsWorkPeriod"                  : StringType(),        # NUMC
    "Tas"                            : StringType(),
    "Betc"                           : StringType(),
    "ModAllow"                       : StringType(),
    "CancelAllow"                    : StringType(),
    "PayMethod"                      : StringType(),
    "Bpn"                            : StringType(),
    "RepFreq"                        : StringType(),
    "FmfgusKey"                      : StringType(),
    "RfmPsstRule"                    : StringType(),
    "RfmPsstGroup"                   : StringType(),
    "Pargb"                          : StringType(),
    "AufplOaa"                       : StringType(),        # NUMC
    "AplzlOaa"                       : StringType(),        # NUMC
    "Vlcendcu"                       : StringType(),
    "WrfCharstc1"                    : StringType(),
    "WrfCharstc2"                    : StringType(),
    "WrfCharstc3"                    : StringType(),
    "Arsnum"                         : StringType(),        # NUMC
    "Arspos"                         : StringType(),        # NUMC
    "WtyscClmitem"                   : StringType(),
}
 

def ingest_data(start_date: str, end_date: str, write_mode: str = "overwrite"):

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    url = "https://poc-sapdevene.enesis.com:8020/sap/opu/odata/sap/ZCDS_VBAP_ODATA_CDS/ZCDS_VBAP_ODATA"
    username = dbutils.secrets.get(scope="sap-odata-username", key="client_secret")
    password = dbutils.secrets.get(scope="sap-odata-password", key="client_secret")

    headers = {
        "Accept": "application/atom+xml",  # eksplisit minta XML
        "sap-client": "120"
    }

    params = {
                "$filter": f"Erdat ge datetime'{start_date}T00:00:00' and Erdat le datetime'{end_date}T23:59:59'",
            }

    response = requests.get(
        url,
        auth=HTTPBasicAuth(username, password),
        headers=headers,
        params=params,
        verify=False,
        timeout=60
    )
    print(f"Status: {response.status_code}")
    print(f"Content-Type: {response.headers.get('Content-Type')}")
    print(f"Response length: {len(response.content)} bytes")
    print("---RAW---")
    print(response.text)

    # Parse XML
    try:    root = ET.fromstring(response.content)
    except ET.ParseError as e:
        raise RuntimeError(f"Failed to parse XML: {e}")

    # Parse XML
    root = ET.fromstring(response.content)

    ns = {
        "atom": "http://www.w3.org/2005/Atom",
        "m": "http://schemas.microsoft.com/ado/2007/08/dataservices/metadata",
        "d": "http://schemas.microsoft.com/ado/2007/08/dataservices"
    }

    records = []

    # Cek apakah root adalah <feed> (collection) atau <entry> (single)
    root_tag = root.tag.split("}")[-1]

    if root_tag == "feed":
        # Collection — loop semua entry
        for entry in root.findall("atom:entry", ns):
            props = entry.find(".//m:properties", ns)
            if props is not None:
                row = {child.tag.split("}")[-1]: child.text for child in props}
                records.append(row)

    elif root_tag == "entry":
        # Single entity — langsung ambil properties dari root
        props = root.find(".//m:properties", ns)
        if props is not None:
            row = {child.tag.split("}")[-1]: child.text for child in props}
            records.append(row)

    df = pd.DataFrame(records)
    print(f"Total rows: {len(df):,}")
    print(df.T)  # .T biar keliatan semua kolom (karena banyak field)

    spark = SparkSession.builder.getOrCreate()

    # Step 1: pandas df → Spark df
    spark_df = spark.createDataFrame(df)

    # Step 2: Cast schema
    for col_name, col_type in vbap_schema.items():
        if col_name in spark_df.columns:
            if isinstance(col_type, DateType):
                # SAP OData kirim format ISO: "2019-08-23T00:00:00"
                spark_df = spark_df.withColumn(col_name, to_date(col(col_name), "yyyy-MM-dd'T'HH:mm:ss"))
            else:
                spark_df = spark_df.withColumn(col_name, col(col_name).cast(col_type))

    # Step 3: Fix sisa kolom NullType
    for field in spark_df.schema.fields:
        if isinstance(field.dataType, NullType):
            spark_df = spark_df.withColumn(field.name, spark_df[field.name].cast(StringType()))

    # Step 4: Audit columns + write
    spark_df = spark_df \
        .withColumn("_ingestion_timestamp", current_timestamp()) \
        .withColumn("_source", lit("ZCDS_VBAP_ODATA"))

    spark_df.write \
        .format("delta") \
        .mode(write_mode) \
        .option("overwriteSchema", "true") \
        .saveAsTable("poc_enesis.bronze.ZCDS_VBAP_ODATA")

    print("✅ Bronze table berhasil dibuat!")
    spark.sql("SELECT COUNT(*) as total FROM poc_enesis.bronze.ZCDS_VBAP_ODATA").show()

In [0]:
print("========= BATCH 1: Januari - Agustus =========")
ingest_data("2019-01-01", "2019-08-31", write_mode="overwrite") 

In [0]:
print("========= BATCH 2: September =========")
ingest_data("2019-09-01", "2019-09-30", write_mode="append") 

In [0]:
print("========= BATCH 3: Oktober =========")
ingest_data("2019-10-01", "2019-10-31", write_mode="append") 

In [0]:
print("========= BATCH 4: November =========")
ingest_data("2019-11-01", "2019-11-30", write_mode="append") 

In [0]:
print("========= BATCH 5: Desember =========")
ingest_data("2019-12-01", "2019-12-31", write_mode="append") 

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit, to_date

watermark_row = spark.createDataFrame([Row(
    source_name   = "SAP_ODATA",
    table_name    = "ZCDS_VBAP_ODATA",
    last_run_date = RUN_DATE[:10],
    last_run_ts   = None,
    status        = "SUCCESS",
    rows_ingested = int(spark_df.count())
)])

watermark_row = watermark_row \
    .withColumn("last_run_date", to_date("last_run_date")) \
    .withColumn("last_run_ts", current_timestamp())

from delta.tables import DeltaTable
wm = DeltaTable.forName(spark, "poc_enesis.bronze.pipeline_watermark")
wm.alias('t').merge(
    watermark_row.alias('s'),
    't.source_name = s.source_name AND t.table_name = s.table_name'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
print("✅ Watermark updated")


In [0]:
import uuid
from datetime import datetime

run_id    = str(uuid.uuid4())[:8]
started   = datetime.utcnow()
SCHEMA     = 'BRONZE'
TABLE     = 'ZCDS_VBAP_ODATA'

try:
    # ... semua logic ingest & write ke bronze ...
    rows_in = spark_df.count()
    status  = 'SUCCESS'
    msg     = f'{rows_in} rows ingested'
except Exception as e:
    rows_in = 0
    status  = 'FAILED'
    msg     = str(e)[:500]
    raise  # re-raise supaya Job tau pipeline gagal
finally:
    ended = datetime.utcnow()
    log_df = spark.createDataFrame([{
        'run_id': run_id, 'schema_name': SCHEMA, 'table_name': TABLE,
        'status': status, 'message': msg, 'rows_in': rows_in, 'rows_out': 0,
        'started_at': started, 'ended_at': ended
    }])
    log_df.write.format('delta').mode('append').saveAsTable(
        'poc_enesis.bronze.run_log'
    )
